In [ ]:
def _assimilate(x):
    return x if isinstance(x, Value) else Value(x)


# scalar
class Value:
    def __init__(self, data, operands=()):
        self.data = data
        self.grad = 0
        self._operands = set(operands)
        self._backward = None

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = _assimilate(other)
        out = Value(data=self.data + other.data, operands=(self, other))

        def _backward():
            self.grad += out.grad * 1.0
            other.grad += out.grad * 1.0

        out._backward = _backward

        return out

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        other = _assimilate(other)
        out = Value(data=self.data - other.data, operands=(self, other))

        def _backward():
            self.grad += out.grad * 1.0
            other.grad += out.grad * -1.0

        out._backward = _backward

        return out

    def __rsub__(self, other):
        return _assimilate(other) - self

    def __mul__(self, other):
        other = _assimilate(other)
        out = Value(data=self.data * other.data, operands=(self, other))

        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data

        out._backward = _backward

        return out

    def __rmul__(self, other):
        return self * other

    def __neg__(self):
        return self * -1

    def __truediv__(self, other):  # x * y^-1
        other = _assimilate(other)
        out = Value(data=self.data / other.data, operands=(self, other))

        def _backward():
            self.grad += out.grad * (1 / other.data)  # 1 / y
            other.grad += out.grad * (-self.data / (other.data**2))  # - (x / y^2)

        out._backward = _backward

        return out

    def __rtruediv__(self, other):
        return _assimilate(other) / self

    def __pow__(self, other):
        assert isinstance(other, (int, float)), (
            "The B230 can only exponentiate constants (int/float)"
        )
        out = Value(data=self.data**other, operands=(self,))

        def _backward():
            self.grad += out.grad * (other * (self.data ** (other - 1)))

        out._backward = _backward

        return out

    # skipping rpow for now: probably will never use it


In [32]:
x = Value(10)
y = Value(5)
z = x / y

z

Value(data=2.0)

In [33]:
a = Value(10)
c = a ** 5

c

Value(data=100000)

In [34]:
z.grad = 1
z._backward()
c.grad = 1
c._backward()

In [35]:
print(x.grad)
print(y.grad)
print(a.grad)

0.2
-0.4
50000
